# DDA - Label-Free (with FlashLFQ)

This tutorial involves how to analyze DDA LFQ data with combining DB search tools and FlashLFQ (for quantification).

For DDA label-free analysis, sometimes, we need to use stand-alone Label-free quantifcation tools such as [FlashLFQ](https://github.com/smith-chem-wisc/FlashLFQ) to quantify with more detailed options (e.g. MBR).

To utilize FlashLFQ, please find tutorials and installation guides in FlashLFQ documentation. This supports Docker, GUI, and conda environment.


## Data Preparation

### Load Required Pacakages


In [1]:
import msmu as mm
import pandas as pd

### Read Data and PSM Filtering

In this tutorial, we will use [PXD012986](https://www.ebi.ac.uk/pride/archive/projects/PXD012986) (Uszkoreit _et al_., 2022) dataset which is mentioned in [DDA-LFQ](../dda-lfq) tutorial section.

To combine FlashLFQ quantification result with msmu, we need to read PSM result file from DB search tools and filter PSMs based on q-value or other criteria, which is because FlashLFQ assumes that the input PSMs are already filtered.


In [2]:
base_dir = "https://raw.githubusercontent.com/bertis-informatics/msmu/refs/heads/dev/data/sage_lfq"
sage_idents = f"{base_dir}/sage/results.sage.tsv"
sdrf = f"{base_dir}/meta.sdrf.tsv"

mdata = mm.read_sage(identification_file=sage_idents, label="label_free")

# Sample metadata from the SDRF: keep the table on the container, then project it onto obs
mdata = mm.pp.attach_sdrf(mdata, sdrf)
mdata = mm.pp.apply_sdrf_to_obs(mdata)

# Without an SDRF, add_meta() joins any table (DataFrame, csv, tsv, parquet) onto obs instead:
# mdata = mm.pp.add_meta(mdata, sdrf, format="sdrf", metadata_on="assay name")

mdata = mm.pp.add_filter(mdata, modality="psm", column="q_value", keep="lt", value=0.01)
mdata = mm.pp.apply_filter(mdata, modality="psm")

mdata

INFO - Reading SAGE Identification data: 1 file(s)


INFO - Validating SDRF metadata for https://raw.githubusercontent.com/bertis-informatics/msmu/refs/heads/dev/data/sage_lfq/meta.sdrf.tsv.


INFO - SDRF validation succeeded for https://raw.githubusercontent.com/bertis-informatics/msmu/refs/heads/dev/data/sage_lfq/meta.sdrf.tsv.


INFO - Applying var filters for psm: ['q_value_lt_0.01']


MuData object with n_obs × n_vars = 6 × 4336
  obs:	'source name', 'characteristics[organism]', 'characteristics[organism part]', 'characteristics[cell line]', 'characteristics[cell type]', 'characteristics[cellosaurus accession]', 'characteristics[cellosaurus name]', 'characteristics[disease]', 'characteristics[biological replicate]', 'assay name', 'technology type', 'comment[proteomexchange accession number]', 'comment[proteomics data acquisition method]', 'comment[fraction identifier]', 'comment[technical replicate]', 'comment[label]', 'comment[instrument]', 'comment[cleavage agent details]', 'factor value[condition]'
  uns:	'_log', 'sdrf'
  1 modality
    psm:	6 × 4336
      obs:	'source name', 'characteristics[organism]', 'characteristics[organism part]', 'characteristics[cell line]', 'characteristics[cell type]', 'characteristics[cellosaurus accession]', 'characteristics[cellosaurus name]', 'characteristics[disease]', 'characteristics[biological replicate]', 'assay name', 'techno

## Export FlashLFQ Input File

After filtering PSMs, we can export the PSMs to FlashLFQ input format using [`mm.io.write_flashlfq_input()`](../../reference/io/write_flashlfq_input/) function.


In [3]:
mm.io.write_flashlfq_input(mdata, "flashlfq_input.tsv")

## (optional in here) Run FlashLFQ

After exporting FlashLFQ input file, we can run FlashLFQ with proper parameters (e.g. MBR) to quantify peptides.

The command line example below shows how to run FlashLFQ in Linux. Please adjust the parameters based on your experimental design and FlashLFQ documentation.

You can skip this step in this tutorial and directly use the provided FlashLFQ quantification result file.


In [4]:
# bash

# dotnet CMD.dll --idt "flashlfq_input.tsv" --rep "/path/to/spectra/directory/" --ppm 5 --chg

# or using Docker

# docker run --rm -v /path/to/local/directory:/data smithchemwisc/flashlfq:1.0.3 \
#     --idt "/data/flashlfq_input.tsv" \
#     --rep "/data/spectra/" \
#     --ppm 5 \
#     --chg

## Attach FlashLFQ result to mdata

Peptide quantification result from FlashLFQ can be attached to `mdata` using [`mm.io.add_quant()`](../../reference/io/add_quant/) function with `quant_tool="flashlfq"` parameter with a file named "QuantifiedPeptides.tsv" containing peptide level quantification values and evidences.


In [5]:
flashlfq_dir = f"https://raw.githubusercontent.com/bertis-informatics/msmu/refs/heads/main/data/flashlfq"
flashlfq_peptides = f"{flashlfq_dir}/QuantifiedPeptides.tsv"

mdata = mm.pp.to_peptide(mdata)

mdata = mm.io.add_quant(mdata, quant_data=flashlfq_peptides, quant_tool="flashlfq")

mdata = mm.pp.log2_transform(mdata, modality="peptide")

mdata

INFO - Peptide-level identifications: 3683 (3664 at 1% FDR)


INFO - Building new peptide quantification data.


INFO - Filled existing peptide modality with flashlfq quantification.


MuData object with n_obs × n_vars = 6 × 8019
  obs:	'source name', 'characteristics[organism]', 'characteristics[organism part]', 'characteristics[cell line]', 'characteristics[cell type]', 'characteristics[cellosaurus accession]', 'characteristics[cellosaurus name]', 'characteristics[disease]', 'characteristics[biological replicate]', 'assay name', 'technology type', 'comment[proteomexchange accession number]', 'comment[proteomics data acquisition method]', 'comment[fraction identifier]', 'comment[technical replicate]', 'comment[label]', 'comment[instrument]', 'comment[cleavage agent details]', 'factor value[condition]'
  uns:	'_log', 'sdrf'
  2 modalities
    psm:	6 × 4336
      obs:	'source name', 'characteristics[organism]', 'characteristics[organism part]', 'characteristics[cell line]', 'characteristics[cell type]', 'characteristics[cellosaurus accession]', 'characteristics[cellosaurus name]', 'characteristics[disease]', 'characteristics[biological replicate]', 'assay name', 'tech